# ProspectML - Data Cleaning and Preparation

This notebook prepares Minor League and Major League baseball datasets for modeling.

The goal is to create a player-level dataset that connects Minor League performance metrics with future Major League outcomes.

The cleaning process includes:
- standardizing player identifiers
- removing unnecessary variables
- resolving duplicate player-team records
- combining Minor League levels
- merging MLB outcomes

In [3]:
import pandas as pd
import numpy as np
import unicodedata
import re

## Import Raw Data

The raw FanGraphs exports are loaded into pandas DataFrames for cleaning and preparation.

In [4]:
milb_career = pd.read_csv(
    "C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\Milb_Raw_Stats_Career.csv",
    encoding="latin1"
)

milb_highA = pd.read_csv(
    "C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\Milb_Raw_Stats_HighA.csv",
    encoding="latin1"
)

milb_AA = pd.read_csv(
    "C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\Milb_Raw_Stats_AA.csv",
    encoding="latin1"
)

milb_AAA = pd.read_csv(
    "C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\Milb_Raw_Stats_AAA.csv",
    encoding="latin1"
)

mlb = pd.read_csv(
    "C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\MLB_Raw_Stats.csv",
    encoding="latin1"
)

## Inspect Dataset Structure

The datasets are reviewed to understand their column structure before cleaning.

In [5]:
milb_career.head()

,#,Name,Team,Level,Age,PA,BB%,K%,BB/K,AVG,...,SLG,OPS,ISO,Spd,BABIP,wSB,wRC,wRAA,wOBA,wRC+
0,1,Kris Bryant,CHC,"A-,A+,AA,AAA,R",21-26,792,12.90%,26.80%,0.48,0.325,...,0.661,1.085,0.336,5.3,0.396,1.7,180,84.7,0.469,188
1,2,Brandon Belt,SFG,"A+,AA,AAA",22-33,873,16.40%,17.90%,0.92,0.347,...,0.603,1.060,0.256,5.9,0.401,-0.5,202,85.7,0.458,177
2,3,Kelvin Diaz,CLE,R,19-21,765,9.70%,12.00%,0.80,0.333,...,0.532,0.958,0.199,5.9,0.356,0.1,163,70.0,0.461,168
3,4,Alex Gordon,KCR,"A+,AA,AAA,R",22-34,1138,14.90%,19.90%,0.75,0.324,...,0.576,1.018,0.252,5.6,0.381,4.2,246,99.7,0.445,167
4,5,Julio Rodríguez,SEA,"A,A+,AA,R",17-20,962,10.20%,18.90%,0.54,0.331,...,0.543,0.955,0.212,7.5,0.391,2.8,200,78.6,0.432,167


In [6]:
mlb.head()

,#,Name,Team,PA,BB%,K%,BB/K,AVG,OBP,SLG,...,BABIP,UBR,wGDP,XBR,wSB,wRC,wRAA,wOBA,wRC+,WAR
0,1,Aaron Judge,NYY,5002,16.30%,27.40%,0.60,0.294,0.413,0.615,...,0.349,6.6,0.9,-6.6,-2.1,1051,454.1,0.425,178,87.1
1,2,Mike Trout,LAA,7203,14.80%,23.10%,0.64,0.294,0.406,0.570,...,0.341,23.9,20.0,7.9,20.4,1392,555.3,0.410,166,64.7
2,3,Yordan Alvarez,HOU,2867,12.10%,19.70%,0.61,0.297,0.389,0.573,...,0.320,-4.7,-1.3,-8.2,-3.0,549,206.4,0.402,163,62.5
3,4,Juan Soto,4 Tms,4803,18.70%,17.30%,1.08,0.282,0.417,0.531,...,0.300,-10.2,0.0,-7.7,0.5,919,344.4,0.402,158,61.6
4,5,Shohei Ohtani,2 Tms,4329,12.50%,25.50%,0.49,0.282,0.374,0.582,...,0.325,3.8,10.4,10.4,9.9,810,293.4,0.397,156,60.4


## Remove Unnecessary MLB Statistics

The MLB dataset contains advanced statistics that are not available in the Minor League datasets.

These variables are removed to maintain consistency between datasets.

In [7]:
mlb = mlb.drop(
    columns=["UBR", "WAR", "XBR", "wGDP"]
)

## Create PlayerID

Player names are standardized into a unique identifier used for merging datasets.

The function:
- removes accents
- converts names to lowercase
- removes punctuation
- preserves suffixes such as Jr., Sr., II, and III

Preserving suffixes prevents different players with similar names from being incorrectly combined.

In [8]:
def create_player_id(name):
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("utf-8")
    name = name.lower()
    
    name = re.sub(r"[^a-z0-9 ]", "", name)
    name = re.sub(r"\s+", "_", name).strip("_")
    
    return name

In [9]:
milb_career["PlayerID"] = milb_career["Name"].apply(create_player_id)

milb_highA["PlayerID"] = milb_highA["Name"].apply(create_player_id)

milb_AA["PlayerID"] = milb_AA["Name"].apply(create_player_id)

milb_AAA["PlayerID"] = milb_AAA["Name"].apply(create_player_id)

mlb["PlayerID"] = mlb["Name"].apply(create_player_id)

## Resolve Multiple Minor League Team Stints

FanGraphs displays separate rows when a player appears for multiple teams during the same season or level.

Because the dataset needs one observation per player, the stint with the highest plate appearances is retained.

This preserves FanGraphs-calculated statistics while keeping the largest available sample size.

In [10]:
milb_career = (
    milb_career
    .sort_values("PA", ascending=False)
    .drop_duplicates("PlayerID")
)

milb_highA = (
    milb_highA
    .sort_values("PA", ascending=False)
    .drop_duplicates("PlayerID")
)

milb_AA = (
    milb_AA
    .sort_values("PA", ascending=False)
    .drop_duplicates("PlayerID")
)

milb_AAA = (
    milb_AAA
    .sort_values("PA", ascending=False)
    .drop_duplicates("PlayerID")
)

## Rename Columns by Level

Each Minor League dataset represents a different development level.

Prefixes are added to statistical columns so that variables remain identifiable after merging.

In [11]:
def add_prefix(df, prefix):
    return df.rename(
        columns={
            col: f"{prefix}_{col.replace('+','_plus')}"
            for col in df.columns
            if col not in ["Name", "PlayerID"]
        }
    )

In [12]:
milb_career = add_prefix(milb_career, "Career")
milb_highA = add_prefix(milb_highA, "HighA")
milb_AA = add_prefix(milb_AA, "AA")
milb_AAA = add_prefix(milb_AAA, "AAA")
mlb = add_prefix(mlb, "MLB")

In [13]:
milb_career = milb_career.drop(
    columns=["Career_Team", "Career_Level"]
)

milb_highA = milb_highA.drop(
    columns=["HighA_Team","HighA_Level"]
)

milb_AA = milb_AA.drop(
    columns=["AA_Team","AA_Level"]
)

milb_AAA = milb_AAA.drop(
    columns=["AAA_Team","AAA_Level"]
)

## Create Master Minor League Dataset

The Minor League datasets are merged using PlayerID.

A left join keeps all players from the career dataset while adding level-specific statistics when available.

In [14]:
milb_master = (
    milb_career
    .merge(milb_highA.drop(columns=["Name"]),
           on="PlayerID",
           how="left")
    .merge(milb_AA.drop(columns=["Name"]),
           on="PlayerID",
           how="left")
    .merge(milb_AAA.drop(columns=["Name"]),
           on="PlayerID",
           how="left")
)

In [15]:
milb_master["PlayerID"].duplicated().sum()

np.int64(0)

## Prepare MLB Outcome Dataset

The MLB dataset represents the player's eventual Major League performance.

Team information is removed because the model evaluates player performance rather than organization.

In [16]:
mlb = mlb.drop(
    columns=["MLB_Team"]
)

In [17]:
mlb[mlb["PlayerID"].duplicated(keep=False)]

,MLB_#,Name,MLB_PA,MLB_BB%,MLB_K%,MLB_BB/K,MLB_AVG,MLB_OBP,MLB_SLG,MLB_OPS,MLB_ISO,MLB_Spd,MLB_BABIP,MLB_wSB,MLB_wRC,MLB_wRAA,MLB_wOBA,MLB_wRC_plus,PlayerID
419,420,Brian Anderson,2527,9.40%,24.30%,0.39,0.251,0.336,0.403,0.738,0.152,3.5,0.316,-2.8,317,12.4,0.322,103,brian_anderson
1245,1246,Brian Anderson,848,7.70%,23.10%,0.33,0.229,0.295,0.370,0.664,0.141,3.7,0.280,-4.0,77,-26.6,0.293,69,brian_anderson


## Remove Ambiguous Player Records

Some players share identical names, creating ambiguity when matching datasets using PlayerID.

Because the goal of this project is to create an accurate player-level dataset, ambiguous player records are removed rather than risking incorrect Minor League and Major League matches.

The only identified collision was Brian Anderson, so both records are removed from the Minor League and Major League datasets before merging.

In [18]:
milb_master = milb_master[
    milb_master["PlayerID"] != "brian_anderson"
]

mlb = mlb[
    mlb["PlayerID"] != "brian_anderson"
]

In [19]:
mlb["PlayerID"].duplicated().sum()

np.int64(0)

## Create Final ProspectML Dataset

The MLB and Minor League datasets are combined using PlayerID.

An inner join is used because the model requires players with both Minor League features and Major League outcomes.

In [20]:
prospectml_master = milb_master.merge(
    mlb.drop(columns=["Name"]),
    on="PlayerID",
    how="inner"
)

In [21]:
prospectml_master["PlayerID"].duplicated().sum()

np.int64(0)

In [22]:
prospectml_master.shape

(936, 91)

In [23]:
prospectml_master.to_csv(
    "C:\\Users\\ncodi\\OneDrive\\Documents\\ProspectML\\Data\\prospectml_master.csv",
    index=False
)